# llms

> Generic LLM-calling utilities (models, prompting, tool schemas) -- deliberately kept independent of boopiter's own Notebook/Cell model, so this module never imports from `cells.py`. `cells.py` imports from here, never the other way around.

In [ ]:
#| default_exp llms

In [ ]:
#| export
from fastcore.utils import *
from lisette import *

In [ ]:
#| export
# slmn (github.com/drscotthawley/slmn, a separate general-purpose toolkit -- see
# boopiter/pyproject.toml for the git dependency) is imported as plain modules here, not
# re-exported via `import *` -- see get_tool_list() below, which assembles actual tool
# functions on demand instead of merging namespaces.
import slmn.nbtools as _slmn_nbtools
import slmn.misc as _slmn_misc
import slmn.remote as _slmn_remote

# slmn.remote also has remote_launch/remote_status/remote_smoke_test, which can run arbitrary
# commands on a remote host over ssh -- a much bigger capability than the rest of slmn's tools.
# Least-privilege: don't hand those to the LLM by default, just the read-only/informational ones.
_SLMN_REMOTE_SAFE = ('fetch_url', 'check_ci', 'remote_gpu_free')

_LOCAL_TOOLS = []  # boopiter-defined tools always available, independent of slmn; empty for now, add as needed

In [ ]:
#| export
def get_tool_list(include_slmn:bool=True # also include slmn's tools -- nbtools + misc (all of it) plus remote's safe/read-only subset (see _SLMN_REMOTE_SAFE; remote_launch/status/smoke_test, which can run arbitrary commands over ssh, are excluded from this default set)
                   ) -> list:
    "Assemble the list of tool functions to offer an LLM: boopiter's own local tools (_LOCAL_TOOLS, currently empty), plus by default most of what slmn publishes. Returns actual callables, not names -- pass straight to prompt_llm(tools=...). Per-notebook ad-hoc tools (see add_tool()) are layered on top of this by the caller, not included here."
    tools = list(_LOCAL_TOOLS)
    if include_slmn:
        for mod in (_slmn_nbtools, _slmn_misc):
            tools += [getattr(mod, name) for name in mod.__all__]
        tools += [getattr(_slmn_remote, name) for name in _SLMN_REMOTE_SAFE]
    return tools

In [ ]:
#| export
def get_ollama_list(): 
    "Get a list of supported ollama models; returns [] and warns if Ollama unavailable."
    import httpx, warnings
    try:
        models = httpx.get("http://localhost:11434/api/tags").json()
        return ['ollama/'+m['model'] for m in models.get('models', [])]
    except Exception as e:
        warnings.warn(f"Ollama not available: {e}")
        return []

In [ ]:
#| export
def get_model_list(): 
    "wrapper routine to get list of all available models from all sources"
    return get_ollama_list()  # TODO: add more model source, e.g. cloud, fileio

In [ ]:
#| eval: false
get_model_list()

['ollama/qwen2.5-coder:latest',
 'ollama/gemma3:4b',
 'ollama/llama3.1:latest',
 'ollama/qwen2.5:latest']

In [ ]:
#| export
# lisette + local (Ollama) models: passing non-empty `tools=` combined with `tool_choice='none'`
# triggers a bug in litellm's MCP-handler codepath that returns a raw dict instead of a proper
# response object (AttributeError: 'dict' object has no attribute 'choices'). Same issue hit by
# SBrewer15/CellMate (https://github.com/SBrewer15/CellMate) -- this is their patch, adopted as-is:
# drop tool_schemas for just that one call whenever tool_choice=='none', then restore them after.
_orig_chat_call = Chat._call


In [ ]:
#| export
@patch
def _call(self:Chat, msg:str|None=None, prefill:str|None=None, temp:float|None=None, think:str|None=None,
          search:str|None=None, stream:bool=False, max_steps:int=2, step:int=1, final_prompt:dict|None=None,
          tool_choice:str|None=None, max_tokens:int|None=None, **kwargs):
    "Internal method that always yields responses -- patched (see the comment above) to avoid a litellm/Ollama tool-calling bug."
    _orig_tools = self.tool_schemas
    if tool_choice == 'none': self.tool_schemas, tool_choice = None, None
    try: yield from _orig_chat_call(self, msg, prefill, temp, think, search, stream, max_steps, step, final_prompt, tool_choice, max_tokens, **kwargs)
    finally: self.tool_schemas = _orig_tools

In [ ]:
#| export
def _reply_details_html(response, msg) -> str:
    "A collapsible <details> block summarizing an LLM call's metadata (model, finish reason, token counts, tool calls, reasoning) -- not part of the reply's actual content, kept out of Cell.source (see Cell.details) so it's never sent back to the model as context, and shown collapsed, in gray, above the real text. onclick=stopPropagation keeps a click on <summary> from also bubbling into the cell's click-anywhere-to-edit handler."
    u = response.usage
    rows = [('Model', response.model), ('Finish reason', response.choices[0].finish_reason)]
    if u: rows.append(('Tokens', f'{u.prompt_tokens} prompt + {u.completion_tokens} completion = {u.total_tokens} total'))
    if msg.tool_calls: rows.append(('Tool calls', ', '.join(tc.function.name for tc in msg.tool_calls)))
    items = ''.join(f'<li>{k}: {v}</li>' for k, v in rows)
    reasoning = f'<pre style="white-space:pre-wrap">{msg.reasoning_content}</pre>' if getattr(msg, 'reasoning_content', None) else ''
    return (f'<details class="text-gray-400" onclick="event.stopPropagation()"><summary>Reply details</summary>'
            f'<ul>{items}</ul>{reasoning}</details>')

def prompt_llm(context:str, model:str='ollama/qwen2.5-coder:latest', tools:list|None=None) -> tuple[str,str]:
    "Send a prompt to the LLM; returns (content, details_html) -- the reply text itself, and a separate collapsible <details> block of call metadata (model/tokens/finish reason/reasoning) meant to be stored apart from the reply (see Cell.details), not mixed into it. TODO: can we stream the response rather than wait for as a final big chunk?"
    # _skip_mcp_handler avoids litellm's MCP-proxy import chain (needs fastapi/orjson) that we don't use.
    # Drop it (and re-add fastapi/orjson to pyproject.toml) if/when we actually want MCP tool support.
    chat = Chat(model, tools=tools or [], callkw={'_skip_mcp_handler': True}) # FYI: this makes a fresh stateless context each time. is that what we want?
    response = chat(context)
    msg = contents(response)
    return msg.content, _reply_details_html(response, msg)


In [ ]:
#| export
_PREFERRED_MODEL_SUBSTR = 'qwen2.5-coder'  # used if present, regardless of exact tag/version


In [ ]:
#| eval: false
s ="Today is July 18. Who's one famous person with this birthday?"
c = prompt_llm(s) 
print(str(c[0]))
s = """
    Tell me the previous question I asked you, from the previous prompt. 
    I want to see if you retain state between calls"""
c = prompt_llm(s) 
print(str(c[0]))

One famous person born on July 18th is Mark Zuckerberg, the co-founder and CEO of Facebook.
I'm sorry for any confusion, but as an AI language model, I don't have the capability to remember or retain information across separate interactions. Each response is generated independently based on the input provided in each session. If you have a specific question or need assistance with something particular, feel free to ask!


### Tool Use

Example tool from lisette docs:

In [ ]:
def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
#| eval: false
res = prompt_llm("What's 47 + 23? Use the tool.", tools=[add_numbers])
print(res[0])

The sum of 47 and 23 is 70. I have completed my task as requested. If you need further assistance or have additional questions, feel free to ask!


In [ ]:
#| eval: false
# Confirms the re-export actually works: a fresh `import *` from boopiter.llms alone (no direct
# slmn import) should carry slmn's tools straight through.
from boopiter.llms import get_tool_list
tools = get_tool_list()
print(len(tools), "tools:", [t.__name__ for t in tools])